# HadISD 2024 Full-Year Data Cleaning and Quality Assessment

This notebook is the complete replacement for the HadISD cleaning part of the project.  
It works with the large full-year 2024 HadISD CSV file and produces:

- a cleaned HadISD pressure CSV file,
- summary tables for the report,
- report-ready figures for data cleaning and QC.

**Important:** keep the large raw CSV and the cleaned CSV local unless the team explicitly wants to push them. The figures and small summary tables are safe to use in the report.

In [ ]:
# Imports and configuration
from pathlib import Path
from collections import defaultdict
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()

# ------------------------------------------------------------------
# CSV selection
# ------------------------------------------------------------------
# Preferred file names. If none of these exists, the notebook automatically
# selects the largest CSV in this folder, excluding known summary/co-location files.
PREFERRED_CSV_NAMES = [
    "hadisd_adriatic_2024.csv",
    "HadISD_2024.csv",
    "hadisd_2024.csv",
    "hadisd_full_2024.csv",
]

CSV_PATH = None
for name in PREFERRED_CSV_NAMES:
    p = BASE_DIR / name
    if p.exists():
        CSV_PATH = p
        break

if CSV_PATH is None:
    excluded_keywords = ["colocation", "station_qc", "summary", "cleaned", "qc_flagged"]
    candidates = [
        p for p in BASE_DIR.glob("*.csv")
        if not any(k in p.name.lower() for k in excluded_keywords)
    ]
    if not candidates:
        raise FileNotFoundError(
            "No suitable CSV file found in this folder. Put the full-year HadISD CSV in DataAggregation/HadISD."
        )
    CSV_PATH = max(candidates, key=lambda p: p.stat().st_size)

print("Selected CSV:", CSV_PATH.name)
print("File size MB:", round(CSV_PATH.stat().st_size / 1024 / 1024, 2))

# ------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------
FIG_DIR = BASE_DIR / "reports" / "figures" / "hadisd_2024_full_year_qc"
TABLE_DIR = BASE_DIR / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = TABLE_DIR / "hadisd_2024_cleaned_pressure.csv"
SUMMARY_CSV = TABLE_DIR / "hadisd_2024_cleaning_summary.csv"
STATION_SUMMARY_CSV = TABLE_DIR / "hadisd_2024_station_summary.csv"
MONTHLY_SUMMARY_CSV = TABLE_DIR / "hadisd_2024_monthly_summary.csv"
STATION_MONTH_CSV = TABLE_DIR / "hadisd_2024_station_month_availability.csv"
MAX_GAP_CSV = TABLE_DIR / "hadisd_2024_max_temporal_gap_by_station.csv"

# ------------------------------------------------------------------
# Cleaning parameters
# ------------------------------------------------------------------
CHUNKSIZE = 500_000
YEAR = 2024
EXPECTED_HOURLY_OBS = 366 * 24  # 2024 is a leap year: 8784 hours

# Very broad physically plausible pressure range in hPa.
# Values outside this range are considered invalid for the cleaned output.
P_MIN = 850
P_MAX = 1100

# Maximum number of rows kept only for plotting distributions/boxplots.
MAX_SAMPLE_ROWS = 250_000

print("Figures will be saved to:", FIG_DIR)
print("Tables will be saved to:", TABLE_DIR)

## 1. Inspect the CSV schema

This step reads only a few rows to confirm the real column names and data format. It does not load the full file into memory.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

sample_preview = pd.read_csv(CSV_PATH, nrows=10, low_memory=False)

print("Columns:")
for c in sample_preview.columns:
    print("-", c)

print("\nPreview:")
display(sample_preview.head(10))

print("\nData types:")
print(sample_preview.dtypes)

## 2. Chunk-based cleaning and aggregation

The file is large, so the notebook processes it in chunks. The cleaning logic is:

1. keep HadISD pressure observations,
2. parse `dt` as timestamp,
3. convert `value`, `lat`, and `lon` to numeric values,
4. remove invalid essential fields,
5. remove invalid coordinates,
6. remove pressure values outside a broad physical range,
7. remove exact duplicate records,
8. export the cleaned pressure table,
9. compute station/month summaries and QC indicators for the report.

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Strip spaces from column names without changing their meaning.
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def select_hadisd_pressure_rows(df: pd.DataFrame) -> pd.DataFrame:
    # Keep rows corresponding to HadISD pressure observations.
    # The current full-year CSV uses sensor-like columns:
    # sensor_name = HadISD, sensor_code = PRESS, quantity = pressure, value = hPa.
    # The filters are intentionally flexible to avoid failing if one metadata column is missing.
    mask = pd.Series(True, index=df.index)

    if "sensor_name" in df.columns:
        sensor_name = df["sensor_name"].astype(str).str.upper().str.strip()
        if (sensor_name == "HADISD").any():
            mask &= sensor_name == "HADISD"

    if "sensor_code" in df.columns:
        sensor_code = df["sensor_code"].astype(str).str.upper().str.strip()
        if (sensor_code == "PRESS").any():
            mask &= sensor_code == "PRESS"

    if "quantity" in df.columns:
        quantity = df["quantity"].astype(str).str.lower()
        # Only apply this if the column contains pressure-like text.
        if quantity.str.contains("press|pressure|pressione", regex=True, na=False).any():
            mask &= quantity.str.contains("press|pressure|pressione", regex=True, na=False)

    return df.loc[mask].copy()


def make_clean_chunk(chunk: pd.DataFrame):
    # Return cleaned rows, QC rows with flags, and summary counts for a chunk.
    chunk = normalize_columns(chunk)
    raw_rows = len(chunk)

    pressure = select_hadisd_pressure_rows(chunk)
    selected_rows = len(pressure)

    required = ["station_id", "dt", "value", "lat", "lon"]
    missing_required = [c for c in required if c not in pressure.columns]
    if missing_required:
        raise ValueError(f"Missing required columns in CSV: {missing_required}")

    pressure["station_id"] = pressure["station_id"].astype(str).str.strip()
    pressure["dt"] = pd.to_datetime(pressure["dt"], errors="coerce")
    pressure["value"] = pd.to_numeric(pressure["value"], errors="coerce")
    pressure["lat"] = pd.to_numeric(pressure["lat"], errors="coerce")
    pressure["lon"] = pd.to_numeric(pressure["lon"], errors="coerce")

    pressure["flag_missing_station_id"] = pressure["station_id"].isna() | (pressure["station_id"] == "") | (pressure["station_id"].str.lower() == "nan")
    pressure["flag_invalid_datetime"] = pressure["dt"].isna()
    pressure["flag_invalid_value"] = pressure["value"].isna()
    pressure["flag_invalid_coordinates"] = (
        pressure["lat"].isna() | pressure["lon"].isna() |
        ~pressure["lat"].between(-90, 90) |
        ~pressure["lon"].between(-180, 180)
    )
    pressure["flag_not_target_year"] = pressure["dt"].dt.year.ne(YEAR)
    pressure["flag_pressure_out_of_range"] = ~pressure["value"].between(P_MIN, P_MAX)

    # Exact duplicates are checked after parsing essential fields.
    before_dup = len(pressure)
    duplicate_subset = ["station_id", "dt"]
    if "sensor_code" in pressure.columns:
        duplicate_subset.append("sensor_code")
    else:
        duplicate_subset.append("value")
    pressure = pressure.drop_duplicates(subset=duplicate_subset)
    duplicates_removed = before_dup - len(pressure)

    # Clean rows: remove invalid essentials and physically impossible pressure values.
    clean_mask = ~(
        pressure["flag_missing_station_id"] |
        pressure["flag_invalid_datetime"] |
        pressure["flag_invalid_value"] |
        pressure["flag_invalid_coordinates"] |
        pressure["flag_not_target_year"] |
        pressure["flag_pressure_out_of_range"]
    )
    clean = pressure.loc[clean_mask].copy()

    # Standard analysis columns.
    clean["date"] = clean["dt"].dt.date
    clean["month"] = clean["dt"].dt.month

    # Keep useful columns only, if they exist.
    keep_cols = [
        "station_id", "dt", "date", "month", "value", "lat", "lon", "unit",
        "sensor_name", "sensor_code", "quantity", "quota", "gestore", "provincia",
        "codseqst", "point", "aggiornamento"
    ]
    keep_cols = [c for c in keep_cols if c in clean.columns]
    clean = clean[keep_cols]

    summary = {
        "raw_rows": raw_rows,
        "selected_pressure_rows": selected_rows,
        "missing_station_id": int(pressure["flag_missing_station_id"].sum()),
        "invalid_datetime": int(pressure["flag_invalid_datetime"].sum()),
        "invalid_value": int(pressure["flag_invalid_value"].sum()),
        "invalid_coordinates": int(pressure["flag_invalid_coordinates"].sum()),
        "not_target_year": int(pressure["flag_not_target_year"].sum()),
        "pressure_out_of_range": int(pressure["flag_pressure_out_of_range"].sum()),
        "duplicates_removed": int(duplicates_removed),
        "clean_rows": int(len(clean)),
    }

    return clean, pressure, summary

print("Cleaning functions ready.")

In [ ]:
# Main cleaning pass
summary_totals = defaultdict(int)
monthly_counts = defaultdict(int)
daily_sum = defaultdict(float)
daily_count = defaultdict(int)
station_stats = defaultdict(lambda: {
    "n_obs": 0,
    "lat_sum": 0.0,
    "lon_sum": 0.0,
    "lat_count": 0,
    "lon_count": 0,
    "min_dt": None,
    "max_dt": None,
    "pressure_sum": 0.0,
    "pressure_count": 0,
    "pressure_min": np.inf,
    "pressure_max": -np.inf,
})
station_month_counts = defaultdict(int)
station_qc_counts = defaultdict(lambda: defaultdict(int))
plot_samples = []

# Remove previous cleaned output to avoid appending to an old file.
if CLEANED_CSV.exists():
    CLEANED_CSV.unlink()

first_write = True
chunk_counter = 0

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNKSIZE, low_memory=False):
    chunk_counter += 1
    clean, flagged, summary = make_clean_chunk(chunk)

    for k, v in summary.items():
        summary_totals[k] += int(v)

    # Save cleaned output progressively.
    if len(clean) > 0:
        clean.to_csv(CLEANED_CSV, mode="w" if first_write else "a", header=first_write, index=False)
        first_write = False

        # Monthly counts
        for m, n in clean.groupby("month").size().items():
            monthly_counts[int(m)] += int(n)

        # Daily sums/counts
        daily_group = clean.groupby("date")["value"].agg(["sum", "count"])
        for d, row in daily_group.iterrows():
            daily_sum[d] += float(row["sum"])
            daily_count[d] += int(row["count"])

        # Station statistics
        grouped = clean.groupby("station_id")
        for sid, g in grouped:
            st = station_stats[sid]
            n = len(g)
            st["n_obs"] += int(n)
            st["lat_sum"] += float(g["lat"].sum())
            st["lon_sum"] += float(g["lon"].sum())
            st["lat_count"] += int(g["lat"].notna().sum())
            st["lon_count"] += int(g["lon"].notna().sum())
            st["pressure_sum"] += float(g["value"].sum())
            st["pressure_count"] += int(g["value"].notna().sum())
            st["pressure_min"] = min(st["pressure_min"], float(g["value"].min()))
            st["pressure_max"] = max(st["pressure_max"], float(g["value"].max()))
            g_min = g["dt"].min()
            g_max = g["dt"].max()
            st["min_dt"] = g_min if st["min_dt"] is None else min(st["min_dt"], g_min)
            st["max_dt"] = g_max if st["max_dt"] is None else max(st["max_dt"], g_max)

        # Station-month counts
        sm = clean.groupby(["station_id", "month"]).size()
        for (sid, m), n in sm.items():
            station_month_counts[(sid, int(m))] += int(n)

        # Plot samples, random small sample from each chunk.
        if sum(len(s) for s in plot_samples) < MAX_SAMPLE_ROWS:
            n_sample = min(5000, len(clean))
            if n_sample > 0:
                plot_samples.append(clean[["station_id", "dt", "month", "value", "lat", "lon"]].sample(n=n_sample, random_state=chunk_counter))

    # QC counts by station from all selected pressure rows.
    if len(flagged) > 0 and "station_id" in flagged.columns:
        flag_cols = [
            "flag_missing_station_id", "flag_invalid_datetime", "flag_invalid_value",
            "flag_invalid_coordinates", "flag_not_target_year", "flag_pressure_out_of_range"
        ]
        valid_sid = flagged[~flagged["station_id"].isna()].copy()
        if len(valid_sid) > 0:
            qg = valid_sid.groupby("station_id")[flag_cols].sum(numeric_only=True)
            for sid, row in qg.iterrows():
                for fc in flag_cols:
                    station_qc_counts[sid][fc] += int(row[fc])

    if chunk_counter % 5 == 0:
        print(f"Processed {chunk_counter} chunks | clean rows so far: {summary_totals['clean_rows']:,}")

print("\nCleaning completed.")
print("Chunks processed:", chunk_counter)
print("Cleaned CSV saved to:", CLEANED_CSV)
print("Clean rows:", f"{summary_totals['clean_rows']:,}")

## 3. Save summary tables

In [ ]:
# Cleaning summary table
summary_df = pd.DataFrame([dict(summary_totals)])
summary_df["input_file"] = CSV_PATH.name
summary_df["year"] = YEAR
summary_df["pressure_min_hpa"] = P_MIN
summary_df["pressure_max_hpa"] = P_MAX
summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df.T.rename(columns={0: "value"}))

# Monthly summary
monthly_summary = pd.DataFrame({
    "month": list(range(1, 13)),
    "n_observations": [monthly_counts.get(m, 0) for m in range(1, 13)]
})
monthly_summary["month_name"] = pd.to_datetime(monthly_summary["month"], format="%m").dt.month_name()
monthly_summary = monthly_summary[["month", "month_name", "n_observations"]]
monthly_summary.to_csv(MONTHLY_SUMMARY_CSV, index=False)

display(monthly_summary)

# Station summary
station_rows = []
for sid, st in station_stats.items():
    n_obs = st["n_obs"]
    station_rows.append({
        "station_id": sid,
        "n_observations": n_obs,
        "completeness_pct": 100 * n_obs / EXPECTED_HOURLY_OBS,
        "lat": st["lat_sum"] / st["lat_count"] if st["lat_count"] else np.nan,
        "lon": st["lon_sum"] / st["lon_count"] if st["lon_count"] else np.nan,
        "first_timestamp": st["min_dt"],
        "last_timestamp": st["max_dt"],
        "pressure_mean_hpa": st["pressure_sum"] / st["pressure_count"] if st["pressure_count"] else np.nan,
        "pressure_min_hpa": st["pressure_min"] if np.isfinite(st["pressure_min"]) else np.nan,
        "pressure_max_hpa": st["pressure_max"] if np.isfinite(st["pressure_max"]) else np.nan,
    })

station_summary = pd.DataFrame(station_rows).sort_values("n_observations", ascending=False)
station_summary.to_csv(STATION_SUMMARY_CSV, index=False)

display(station_summary.head(20))

# Station-month availability matrix
station_month_df = pd.DataFrame(
    [(sid, m, n) for (sid, m), n in station_month_counts.items()],
    columns=["station_id", "month", "n_observations"]
)
station_month_matrix = station_month_df.pivot_table(
    index="station_id", columns="month", values="n_observations", fill_value=0
)
station_month_matrix = station_month_matrix.reindex(columns=list(range(1, 13)), fill_value=0)
station_month_matrix.to_csv(STATION_MONTH_CSV)

display(station_month_matrix.head(20))

## 4. Temporal gap analysis

This step computes the maximum time gap between consecutive observations for each station using the cleaned CSV. It may take a little time because it reads timestamp information from the cleaned file.

In [ ]:
def compute_max_gaps(cleaned_csv: Path) -> pd.DataFrame:
    if not cleaned_csv.exists() or cleaned_csv.stat().st_size == 0:
        return pd.DataFrame(columns=["station_id", "max_gap_hours", "median_gap_hours"])

    # Read only the two columns needed for gap analysis.
    gap_df = pd.read_csv(cleaned_csv, usecols=["station_id", "dt"], parse_dates=["dt"])
    gap_df = gap_df.dropna(subset=["station_id", "dt"])
    gap_df["station_id"] = gap_df["station_id"].astype(str)
    gap_df = gap_df.sort_values(["station_id", "dt"])
    gap_df["gap_hours"] = gap_df.groupby("station_id")["dt"].diff().dt.total_seconds() / 3600

    out = gap_df.groupby("station_id")["gap_hours"].agg(
        max_gap_hours="max",
        median_gap_hours="median"
    ).reset_index()
    return out

try:
    gap_summary = compute_max_gaps(CLEANED_CSV)
    gap_summary.to_csv(MAX_GAP_CSV, index=False)
    display(gap_summary.sort_values("max_gap_hours", ascending=False).head(20))
except MemoryError:
    gap_summary = pd.DataFrame(columns=["station_id", "max_gap_hours", "median_gap_hours"])
    print("MemoryError: gap analysis skipped. The other cleaning outputs are still valid.")

print("Gap summary saved to:", MAX_GAP_CSV)

## 5. Generate report-ready figures

In [ ]:
# Prepare plotting data
if plot_samples:
    sample_df = pd.concat(plot_samples, ignore_index=True)
    if len(sample_df) > MAX_SAMPLE_ROWS:
        sample_df = sample_df.sample(n=MAX_SAMPLE_ROWS, random_state=42)
else:
    sample_df = pd.DataFrame(columns=["station_id", "dt", "month", "value", "lat", "lon"])

# Daily mean pressure
daily_df = pd.DataFrame({
    "date": list(daily_sum.keys()),
    "pressure_sum": [daily_sum[d] for d in daily_sum.keys()],
    "pressure_count": [daily_count[d] for d in daily_sum.keys()],
})
if len(daily_df) > 0:
    daily_df["date"] = pd.to_datetime(daily_df["date"])
    daily_df["daily_mean_pressure_hpa"] = daily_df["pressure_sum"] / daily_df["pressure_count"]
    daily_df = daily_df.sort_values("date")

print("Sample rows for distribution plots:", len(sample_df))
print("Daily rows:", len(daily_df))

In [ ]:
def save_current_fig(filename: str):
    path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

# 01 Station map
if len(station_summary) > 0:
    plt.figure(figsize=(8, 7))
    plt.scatter(station_summary["lon"], station_summary["lat"], s=35, alpha=0.8, edgecolors="black", linewidths=0.3)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("HadISD 2024 station coverage")
    plt.grid(True, alpha=0.3)
    save_current_fig("01_hadisd_station_map.png")

# 02 Monthly observation counts
plt.figure(figsize=(10, 5))
plt.bar(monthly_summary["month_name"], monthly_summary["n_observations"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cleaned observations")
plt.title("Monthly HadISD pressure observation counts - 2024")
plt.grid(axis="y", alpha=0.3)
save_current_fig("02_monthly_observation_counts.png")

# 03 Station-month availability heatmap
if len(station_month_matrix) > 0:
    top_stations = station_summary.head(50)["station_id"].astype(str).tolist()
    heat = station_month_matrix.reindex(top_stations).fillna(0)
    plt.figure(figsize=(10, max(6, 0.18 * len(heat))))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label="Number of observations")
    plt.xticks(ticks=np.arange(12), labels=[str(m) for m in range(1, 13)])
    plt.yticks(ticks=np.arange(len(heat.index)), labels=heat.index)
    plt.xlabel("Month")
    plt.ylabel("Station ID")
    plt.title("Station-month availability heatmap - top 50 stations")
    save_current_fig("03_station_month_availability_heatmap.png")

# 04 Pressure distribution
if len(sample_df) > 0:
    plt.figure(figsize=(9, 5))
    plt.hist(sample_df["value"].dropna(), bins=60)
    plt.xlabel("Pressure (hPa)")
    plt.ylabel("Frequency")
    plt.title("Distribution of cleaned HadISD pressure observations")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("04_pressure_distribution.png")

# 05 Pressure boxplot by month
if len(sample_df) > 0:
    box_data = [sample_df.loc[sample_df["month"] == m, "value"].dropna().values for m in range(1, 13)]
    plt.figure(figsize=(10, 5))
    plt.boxplot(box_data, labels=[str(m) for m in range(1, 13)], showfliers=False)
    plt.xlabel("Month")
    plt.ylabel("Pressure (hPa)")
    plt.title("Monthly pressure distribution after cleaning")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("05_pressure_boxplot_by_month.png")

# 06 Daily mean pressure time series
if len(daily_df) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(daily_df["date"], daily_df["daily_mean_pressure_hpa"])
    plt.xlabel("Date")
    plt.ylabel("Daily mean pressure (hPa)")
    plt.title("Daily mean HadISD pressure during 2024")
    plt.grid(True, alpha=0.3)
    save_current_fig("06_daily_mean_pressure_timeseries.png")

# 07 Station completeness distribution
if len(station_summary) > 0:
    plt.figure(figsize=(9, 5))
    plt.hist(station_summary["completeness_pct"].dropna(), bins=30)
    plt.xlabel("Completeness (%)")
    plt.ylabel("Number of stations")
    plt.title("Station-level completeness distribution")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("07_station_completeness_distribution.png")

# 08 QC flags by station
qc_rows = []
for sid, counts in station_qc_counts.items():
    total_flags = sum(int(v) for v in counts.values())
    qc_rows.append({"station_id": sid, "total_qc_flags": total_flags, **counts})
qc_by_station = pd.DataFrame(qc_rows)
if len(qc_by_station) > 0:
    qc_by_station = qc_by_station.sort_values("total_qc_flags", ascending=False)
    qc_by_station.to_csv(TABLE_DIR / "hadisd_2024_qc_flags_by_station.csv", index=False)

    top_qc = qc_by_station.head(20)
    plt.figure(figsize=(10, 5))
    plt.bar(top_qc["station_id"].astype(str), top_qc["total_qc_flags"])
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Number of QC flags")
    plt.title("Stations with the highest number of QC flags")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("08_qc_flags_by_station.png")

# 09 Max temporal gap by station
if len(gap_summary) > 0 and "max_gap_hours" in gap_summary.columns:
    top_gap = gap_summary.sort_values("max_gap_hours", ascending=False).head(20)
    plt.figure(figsize=(10, 5))
    plt.bar(top_gap["station_id"].astype(str), top_gap["max_gap_hours"])
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Maximum temporal gap (hours)")
    plt.title("Largest temporal gaps by station")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("09_max_temporal_gap_by_station.png")

# 10 Cleaning summary bar chart
cleaning_metrics = {
    "Clean rows": summary_totals.get("clean_rows", 0),
    "Invalid timestamps": summary_totals.get("invalid_datetime", 0),
    "Invalid values": summary_totals.get("invalid_value", 0),
    "Invalid coordinates": summary_totals.get("invalid_coordinates", 0),
    "Out-of-range pressure": summary_totals.get("pressure_out_of_range", 0),
    "Duplicates removed": summary_totals.get("duplicates_removed", 0),
}
plt.figure(figsize=(10, 5))
plt.bar(list(cleaning_metrics.keys()), list(cleaning_metrics.values()))
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of rows")
plt.title("HadISD 2024 cleaning summary")
plt.grid(axis="y", alpha=0.3)
save_current_fig("10_cleaning_summary_bar.png")

## 6. Final output summary

In [ ]:
print("Notebook completed successfully.")
print("\nMain outputs:")
print("Cleaned CSV:", CLEANED_CSV)
print("Cleaning summary:", SUMMARY_CSV)
print("Station summary:", STATION_SUMMARY_CSV)
print("Monthly summary:", MONTHLY_SUMMARY_CSV)
print("Station-month availability:", STATION_MONTH_CSV)
print("Max temporal gaps:", MAX_GAP_CSV)
print("Figures folder:", FIG_DIR)

print("\nImportant note:")
print("The raw CSV and cleaned CSV can be large. Do not push them to GitHub unless the team explicitly wants them.")
print("For the report, usually push/use the notebook, summary tables, and PNG figures only.")

## Report interpretation

This notebook supports the report section **HadISD 2024 Data Cleaning and Quality Assessment**.  
It should be described as HadISD full-year pressure-data cleaning, not as a full HadISD–ERA5 comparison, because this CSV does not contain ERA5 columns.

Recommended report outputs:

- station coverage map,
- monthly observation counts,
- station-month availability heatmap,
- pressure distribution,
- monthly pressure boxplots,
- daily mean pressure time series,
- station completeness distribution,
- QC flags by station,
- maximum temporal gaps by station.